# Standard Strategy Optimization Example

This notebook demonstrates parameter optimization for a classic indicator-based strategy using `trade_lab.optimization.OptunaOptimizer`.

In [7]:
from pathlib import Path
from decimal import Decimal, getcontext
import sys

getcontext().prec =3

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'examples' else Path.cwd().resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from trade_lab.backtesting import BacktestEngine
from trade_lab.indicators import EMA, RSI
from trade_lab.optimization import OptunaOptimizer, IntParam, FloatParam
from trade_lab.strategies import StandardStrategy

## 1) Fetch data and split train/validation

In [8]:
data_engine = BacktestEngine(ticker='^GSPC', start='2015-01-01', end='2025-01-01')
full_df = data_engine.fetch_data()

train_df = full_df[:'2021-12-31']
val_df = full_df['2022-01-01':]

print('Rows total:', len(full_df))
print('Train rows:', len(train_df), '| Val rows:', len(val_df))

[*********************100%***********************]  1 of 1 completed

Rows total: 2516
Train rows: 1763 | Val rows: 753


## 2) Define strategy factory and search space

In [9]:
def strategy_factory(params):
    return StandardStrategy(
        indicators=[
            (EMA(period=params['fast']), params['w_fast']),
            (EMA(period=params['slow']), params['w_slow']),
            (RSI(period=params['rsi']), params['w_rsi']),
        ],
        entry_threshold=params['entry_thr'],
        exit_threshold=0.05,
        allow_long=True,
        allow_short=True,
    )

param_space = [
    IntParam('fast', 5, 40),
    IntParam('slow', 20, 150, step=5),
    IntParam('rsi', 7, 28),
    FloatParam('w_fast', Decimal(0.1), Decimal(3.0)),
    FloatParam('w_slow', Decimal(-3.0), Decimal(-0.1)),
    FloatParam('w_rsi', Decimal(0.1), Decimal(2.0)),
    FloatParam('entry_thr', Decimal(0.1), Decimal(0.8)),
]

## 3) Run optimization

In [10]:
optimizer = OptunaOptimizer(
    strategy_factory=strategy_factory,
    param_space=param_space,
    train_df=train_df,
    val_df=val_df,
    metric='sharpe_ratio',
    n_trials=50,
    n_jobs=1,
)

result = optimizer.optimize()
print(result.summary())

Best trial: 0. Best value: 6.4233e+08: 100%|██████████| 50/50 [00:01<00:00, 34.72it/s]


  Optimisation Result
  Metric     : sharpe_ratio (maximize)
  Best value : 642329958.4450  (train)
  Trials     : 50 completed, 0 failed

  Best parameters:
    fast                           5
    slow                           20
    rsi                            15
    w_fast                         2.6276078557098104
    w_slow                         -0.43056831955334607
    w_rsi                          1.1309243399425986
    entry_thr                      0.43926245722155544

  Validation sharpe_ratio           1537265938.3907


In [11]:
result.trials_df.sort_values('value', ascending=False).head(10)

,trial_number,value,state,duration_s,fast,slow,rsi,w_fast,w_slow,w_rsi,entry_thr
0,0,6.423300e+08,COMPLETE,0.026995,5,20,15,2.627608,-0.430568,1.130924,0.439262
22,22,3.995245e+08,COMPLETE,0.035319,5,75,14,2.557691,-1.243297,0.194098,0.477008
21,21,2.243942e+08,COMPLETE,0.029131,5,65,13,2.474413,-1.055308,0.116274,0.555627
12,12,1.902347e+08,COMPLETE,0.030932,5,60,16,2.395112,-0.785876,0.163476,0.577999
41,41,1.722349e+08,COMPLETE,0.031222,5,60,16,2.397390,-0.834268,0.106879,0.594348
31,31,1.406648e+08,COMPLETE,0.030874,5,60,16,2.458031,-0.806905,0.103676,0.604647
11,11,1.232536e+08,COMPLETE,0.029922,5,65,12,2.364603,-0.954509,0.526784,0.576136
43,43,5.457212e+07,COMPLETE,0.031408,6,55,16,2.832599,-0.682433,0.105809,0.487410
49,49,3.046001e+07,COMPLETE,0.031599,6,45,19,2.922485,-1.018817,0.166764,0.619050
40,40,9.447682e+06,COMPLETE,0.030917,9,85,15,2.263385,-1.634564,0.444863,0.278300


In [12]:
print('Best train metric:', result.best_value)
if result.val_metrics is not None:
    print('Validation sharpe_ratio:', result.val_metrics.get('sharpe_ratio'))

Best train metric: 642329958.4450175
Validation sharpe_ratio: 1537265938.3907156
